## Extended Data Table 3

Enviorment: env 1

In [4]:
import pandas as pd
import pickle
import os
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, roc_curve, auc, precision_recall_curve
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import confusion_matrix
from scipy.stats import sem
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)

### Performance-optimized metrics

In [5]:
# for the meta pkls
# 0: prediction
# 1: probs
# 2: f1
# 3: feature sets
# 4: ground truth labels
# 5: model params

In [6]:
pw1_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw1/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_100,max_depth_2,learning_rate_0.3,subsample_1.0,scale_pos_weight_12,n_feats_16}_oof.pkl"
pw2_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw2/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_100,max_depth_2,learning_rate_0.3,subsample_1.0,scale_pos_weight_12,n_feats_16}_oof.pkl"
pw3_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw3/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_100,max_depth_3,learning_rate_0.1,subsample_1.0,scale_pos_weight_12,n_feats_16}_oof.pkl"
pw4_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw4/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_2,learning_rate_0.3,subsample_1.0,scale_pos_weight_8,n_feats_8}_oof.pkl"
pw5_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw5/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_2,learning_rate_0.1,subsample_0.8,scale_pos_weight_12,n_feats_32}_oof.pkl"
pw6_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw6/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_2,learning_rate_0.3,subsample_1.0,scale_pos_weight_12,n_feats_8}_oof.pkl"
pw7_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw7/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_1,learning_rate_0.1,subsample_0.8,scale_pos_weight_2,n_feats_32}_oof.pkl"
pw8_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw8/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_2,learning_rate_0.1,subsample_0.8,scale_pos_weight_2,n_feats_8}_oof.pkl"
pw9_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw9/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_2,learning_rate_0.1,subsample_1.0,scale_pos_weight_12,n_feats_32}_oof.pkl"
pw10_5 = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/final_pec_hc_pw10/run_20251020/meta_probs_auc_0.5/XGBoost/{n_estimators_250,max_depth_1,learning_rate_0.3,subsample_1.0,scale_pos_weight_8,n_feats_32}_oof.pkl"

all_loo_paths = [pw1_5, pw2_5, pw3_5, pw4_5, pw5_5, pw6_5, pw7_5, pw8_5, pw9_5, pw10_5]

In [7]:
cu_all_probs = []
for i in range(10):
    full_path = all_loo_paths[i]
    with open(full_path, "rb") as f:
        data = pickle.load(f)
    cu_all_probs.append(data[1])
cu_all_probs = pd.DataFrame(cu_all_probs)

In [8]:
clinical = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/clinical_ml_feats_20260430.csv")

In [9]:
fmf_loo = pd.read_csv("/gpfs/data/shenhavlab/users/ca3261/Visionary_AI/data/fmf_pe_results_cu.csv")

In [10]:
fmf_loo["fmf_prob"] = fmf_loo["pe_before_delivery_percent"] / 100
fmf_loo["id"] = [id.lower() for id in fmf_loo["patient_id"]]
fmf_loo = fmf_loo[["fmf_prob", "id"]]

In [11]:
cu_all_probs = cu_all_probs.T
cu_all_probs["id"] = cu_all_probs.index
cu_all_probs = cu_all_probs.reset_index(drop=True)
cu_all_probs = cu_all_probs.merge(clinical, how="left")
cu_all_probs = cu_all_probs.merge(fmf_loo, how="left")

In [12]:
# number subjects with predictions
len(cu_all_probs[~cu_all_probs[0].isna() | ~cu_all_probs[1].isna() | ~cu_all_probs[2].isna() | ~cu_all_probs[3].isna() | ~cu_all_probs[4].isna() | ~cu_all_probs[5].isna() |
    ~cu_all_probs[6].isna() | ~cu_all_probs[7].isna() | ~cu_all_probs[8].isna() | ~cu_all_probs[9].isna()])

1137

In [13]:
# number of pec cases with predictions
len(cu_all_probs[~cu_all_probs[0].isna() & ~cu_all_probs[1].isna() & ~cu_all_probs[2].isna() & ~cu_all_probs[3].isna() & ~cu_all_probs[4].isna() & ~cu_all_probs[5].isna() &
    ~cu_all_probs[6].isna() & ~cu_all_probs[7].isna() & ~cu_all_probs[8].isna() & ~cu_all_probs[9].isna()]) - 82

54

In [14]:
with open("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/final_pec_20251017.pkl", "rb") as f:
    all_pec = pickle.load(f)

In [15]:
cu_all_probs["label"] = [int(id in all_pec) for id in cu_all_probs["id"]]
sum(cu_all_probs["label"])

54

In [16]:
rename_dict = {}

for i in range(10):
    rename_dict[i] = f"pw_{i + 1}"

cu_all_probs = cu_all_probs.rename(columns=rename_dict)

In [17]:
data_path = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data"

In [18]:
sf_cases = f"{data_path}/final_sf_20251028.pkl"
with open(sf_cases, "rb") as f:
    sf_cases = pickle.load(f)
nsf_cases = f"{data_path}/final_nsf_20251028.pkl"
with open(nsf_cases, "rb") as f:
    nsf_cases = pickle.load(f)
eope_cases = f"{data_path}/final_eope_20251017.pkl"
with open(eope_cases, "rb") as f:
    eope_cases = pickle.load(f)
lope_cases = f"{data_path}/final_lope_20251028.pkl"
with open(lope_cases, "rb") as f:
    lope_cases = pickle.load(f)
pec_cases = f"{data_path}/final_pec_20251017.pkl"
with open(pec_cases, "rb") as f:
    pec_cases = pickle.load(f)

In [19]:
pec_data = cu_all_probs[['pw_1', 'pw_2', 'pw_3', 'pw_4', 'pw_5', 'pw_6', 'pw_7', 'pw_8', 'pw_9', 'pw_10', 'label', 'id', "Past PEC", "First Preg", "Age at enrollment"]].copy()
pec_data.loc[:, "pec"] = [id in pec_cases for id in pec_data["id"]]
pec_data.loc[:, "sf"] = [id in sf_cases for id in pec_data["id"]]
pec_data.loc[:, "nsf"] = [id in nsf_cases for id in pec_data["id"]]
pec_data.loc[:, "eope"] = [id in eope_cases for id in pec_data["id"]]
pec_data.loc[:, "lope"] = [id in lope_cases for id in pec_data["id"]]

In [22]:
best_pp = 0.35
best_fp = 0.15
best_age = 37

In [23]:
reweight = cu_all_probs.copy()

for i in range(1, 11):
    pec_data[f"pw_{i}"] = pec_data[f"pw_{i}"] + (reweight["Past PEC"] * best_pp + best_fp * (reweight["First Preg"] * (reweight["Age at enrollment"] > best_age).astype(int)))

In [24]:
data_path = "/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data"

In [25]:
sf_cases = f"{data_path}/final_sf_20251028.pkl"
with open(sf_cases, "rb") as f:
    sf_cases = pickle.load(f)
nsf_cases = f"{data_path}/final_nsf_20251028.pkl"
with open(nsf_cases, "rb") as f:
    nsf_cases = pickle.load(f)
eope_cases = f"{data_path}/final_eope_20251017.pkl"
with open(eope_cases, "rb") as f:
    eope_cases = pickle.load(f)
lope_cases = f"{data_path}/final_lope_20251028.pkl"
with open(lope_cases, "rb") as f:
    lope_cases = pickle.load(f)
pec_cases = f"{data_path}/final_pec_20251017.pkl"
with open(pec_cases, "rb") as f:
    pec_cases = pickle.load(f)

In [26]:
pec_data = cu_all_probs[['pw_1', 'pw_2', 'pw_3', 'pw_4', 'pw_5', 'pw_6', 'pw_7', 'pw_8', 'pw_9', 'pw_10', 'label', 'id', "Past PEC", "First Preg", "Age at enrollment"]].copy()
pec_data.loc[:, "pec"] = [id in pec_cases for id in pec_data["id"]]
pec_data.loc[:, "sf"] = [id in sf_cases for id in pec_data["id"]]
pec_data.loc[:, "nsf"] = [id in nsf_cases for id in pec_data["id"]]
pec_data.loc[:, "eope"] = [id in eope_cases for id in pec_data["id"]]
pec_data.loc[:, "lope"] = [id in lope_cases for id in pec_data["id"]]

In [27]:
for i in range(1, 11):
    pec_data[f"pw_{i}"] = pec_data[f"pw_{i}"] + (reweight["Past PEC"] * best_pp + best_fp * (reweight["First Preg"] * (reweight["Age at enrollment"] > best_age).astype(int)))

In [28]:
cases = ["pec", "sf", "nsf", "eope", "lope"]
pw_results = None
thresh = 0.32
print(thresh)
for c in cases:
    for pw in range(1, 11):
        column = f"pw_{pw}"
        if "pec" == c:
            subset = pec_data[(~pec_data[column].isna())]
        else:
            subset = pec_data[(~pec_data[column].isna()) & ((pec_data[c] & pec_data["pec"]) | ~pec_data["pec"])]
        probabilities = np.array(list(subset[column]))
        predredictions = probabilities > thresh
        grounds = np.array(list(subset["label"]))

        f1 = f1_score(grounds, predredictions)
        tn, fp, fn, tp = confusion_matrix(grounds, predredictions).ravel().tolist()
        f1 = f1_score(grounds, predredictions)
        tpr = tp / (tp + fn) if (tp + fn) else 0
        fpr = fp / (fp + tn) if (fp + tn) else 0
        fnr = fn / (fn + tp) if (fn + tp) else 0
        tnr = tn / (tn + fp) if (tn + fp) else 0
        ppv = tp / (tp + fp) if (tp + fp) else 0
        npv = tn / (tn + fn) if (tn + fn) else 0
        acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0
        auc_stat = roc_auc_score(grounds, probabilities)
        metric_df = pd.DataFrame({
            "cases": [c], "AUC": auc_stat, "PPV": ppv, "NPV": npv, "TPR": tpr, "FPR": fpr,
            'adjusted ppv': tpr * 0.04 / (tpr * 0.04 + (fpr) * (1 - 0.04)) if (tpr * 0.04 + (fpr) * (1 - 0.04)) else 0,
            'adjusted npv': tnr * (1 - 0.04) / (tnr * (1 - 0.04) + fnr * 0.04), "F1": f1
        })
        if pw_results is None:
            pw_results = metric_df
        else:
            pw_results = pd.concat([pw_results, metric_df])
pw_results = pw_results.groupby("cases").mean()
pw_results["order"] = [3, 4, 2, 0, 1]
pw_results = pw_results.sort_values("order")
pw_results.round(2)

0.32


,AUC,PPV,NPV,TPR,FPR,adjusted ppv,adjusted npv,F1,order
cases,,,,,,,,,
pec,0.91,0.64,0.95,0.85,0.15,0.22,0.99,0.73,0
sf,0.90,0.56,0.96,0.84,0.15,0.22,0.99,0.67,1
nsf,0.92,0.37,0.99,0.86,0.15,0.22,0.99,0.51,2
eope,0.93,0.43,0.99,0.90,0.15,0.23,0.99,0.57,3
lope,0.89,0.53,0.96,0.82,0.15,0.21,0.99,0.64,4


In [29]:
cases = ["pec", "sf", "nsf", "eope", "lope"]
pw_results = None
thresh = 0.65
print(thresh)
for c in cases:
    for pw in range(1, 11):
        column = f"pw_{pw}"
        if "pec" == c:
            subset = pec_data[(~pec_data[column].isna())]
        else:
            subset = pec_data[(~pec_data[column].isna()) & ((pec_data[c] & pec_data["pec"]) | ~pec_data["pec"])]
        probabilities = np.array(list(subset[column]))
        predredictions = probabilities > thresh
        grounds = np.array(list(subset["label"]))

        f1 = f1_score(grounds, predredictions)
        tn, fp, fn, tp = confusion_matrix(grounds, predredictions).ravel().tolist()
        f1 = f1_score(grounds, predredictions)
        tpr = tp / (tp + fn) if (tp + fn) else 0
        fpr = fp / (fp + tn) if (fp + tn) else 0
        fnr = fn / (fn + tp) if (fn + tp) else 0
        tnr = tn / (tn + fp) if (tn + fp) else 0
        ppv = tp / (tp + fp) if (tp + fp) else 0
        npv = tn / (tn + fn) if (tn + fn) else 0
        acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0
        auc_stat = roc_auc_score(grounds, probabilities)
        metric_df = pd.DataFrame({
            "cases": [c], "AUC": auc_stat, "PPV": ppv, "NPV": npv, "TPR": tpr, "FPR": fpr,
            'adjusted ppv': tpr * 0.04 / (tpr * 0.04 + (fpr) * (1 - 0.04)) if (tpr * 0.04 + (fpr) * (1 - 0.04)) else 0,
            'adjusted npv': tnr * (1 - 0.04) / (tnr * (1 - 0.04) + fnr * 0.04), "F1": f1
        })
        if pw_results is None:
            pw_results = metric_df
        else:
            pw_results = pd.concat([pw_results, metric_df])
pw_results = pw_results.groupby("cases").mean()
pw_results["order"] = [3, 4, 2, 0, 1]
pw_results = pw_results.sort_values("order")
pw_results.round(2)

0.65


,AUC,PPV,NPV,TPR,FPR,adjusted ppv,adjusted npv,F1,order
cases,,,,,,,,,
pec,0.91,0.81,0.92,0.71,0.05,0.43,0.99,0.75,0
sf,0.90,0.76,0.94,0.71,0.05,0.43,0.99,0.73,1
nsf,0.92,0.58,0.97,0.69,0.05,0.42,0.99,0.62,2
eope,0.93,0.65,0.97,0.75,0.05,0.44,0.99,0.69,3
lope,0.89,0.73,0.94,0.68,0.05,0.42,0.99,0.70,4


### Stability-optimized metrics

#### fmf

In [30]:
fmf_loo = pd.read_csv("/gpfs/data/shenhavlab/users/ca3261/Visionary_AI/data/fmf_pe_results_cu.csv")
fmf_loo["fmf_prob"] = fmf_loo["pe_before_delivery_percent"] / 100
fmf_loo["id"] = [id.lower() for id in fmf_loo["patient_id"]]
fmf_loo = fmf_loo[["fmf_prob", "id"]]

#### maternal information

In [31]:
clinical = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/clinical_ml_feats_20260430.csv")

#### retinal information

In [32]:
all_mets = pd.read_csv("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/rev_strat_pec_hc_pw/run_20260407_stable/meta_probs_auc_0.5/all_pw_metrics.csv")
result = all_mets.loc[all_mets.groupby('pw')['auc'].idxmax(), ['pw', 'hp', 'auc']]

In [33]:
cu_all_probs = []
for pw in range(1, 6):
    hp = result[result["pw"] == pw]["hp"].iloc[0]
    path = f"/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/columbia/rev_strat_pec_hc_pw{pw}/run_20260407_stable/meta_probs_auc_0.5/XGBoost/{hp}_oof.pkl"
    with open(path, "rb") as f:
        data = pickle.load(f)
        cu_all_probs.append(data[1])
cu_all_probs = pd.DataFrame(cu_all_probs)

In [34]:
cu_all_probs = cu_all_probs.T
cu_all_probs["id"] = cu_all_probs.index
cu_all_probs = cu_all_probs.reset_index(drop=True)
cu_all_probs = cu_all_probs.merge(clinical, how="left")
cu_all_probs = cu_all_probs.merge(fmf_loo, how="left")

In [35]:
with open("/gpfs/data/shenhavlab/PROJECTS/visionary_ai/data/clinical_data/final_pec_20251017.pkl", "rb") as f:
    all_pec = pickle.load(f)
cu_all_probs["label"] = [int(id in all_pec) for id in cu_all_probs["id"]]

rename_dict = {}

for i in range(6):
    rename_dict[i] = f"pw_{i + 1}"

cu_all_probs = cu_all_probs.rename(columns=rename_dict)

In [36]:
all_probs = cu_all_probs[~cu_all_probs["pw_1"].isna() | ~cu_all_probs["pw_2"].isna() | ~cu_all_probs["pw_3"].isna() | ~cu_all_probs["pw_4"].isna() | ~cu_all_probs["pw_5"].isna()]
all_probs = all_probs[all_probs["label"] != 1]
len(all_probs) # how many controls

1134

In [37]:
best_pp = 0.35
best_fp = 0.15
best_age = 37

In [38]:
pec_data = cu_all_probs.copy()

In [39]:
clinical_feats = pd.read_csv("/gpfs/data/shenhavlab/users/ca3261/Visionary_AI/data/MFMOphthBearellyWapn-DatasetOphthAAAT2279_DATA_LABELS_2026-03-09_2241.csv")

In [40]:
clinical_feats["id"] = [x.lower() for x in clinical_feats["Study ID"]]
clinical_feats["aspirin"] = clinical_feats["Aspirin?"].str.contains("Yes").fillna(0)
clinical_feats["aspirin"] = [int(x) for x in clinical_feats["aspirin"]]
clinical_feats = clinical_feats[["id", "aspirin"]]

In [41]:
pec_data = pec_data.merge(clinical_feats, how="left", on="id")

In [42]:
import seaborn as sns

In [43]:
cases = ["pec"]
pw_results = None
thresh = 0.65
print(thresh)
for c in cases:
    for pw in range(1, 5):
        column = f"pw_{pw}"
        if "pec" == c:
            subset = pec_data[(~pec_data[column].isna())]
        else:
            subset = pec_data[(~pec_data[column].isna()) & ((pec_data[c] & pec_data["pec"]) | ~pec_data["pec"])]
        probabilities = np.array(list(subset[column])) + (subset["Past PEC"] * best_pp + best_fp * (subset["First Preg"] * (subset["Age at enrollment"] > best_age).astype(int)))
        predredictions = probabilities > thresh
        grounds = np.array(list(subset["label"]))

        f1 = f1_score(grounds, predredictions)
        tn, fp, fn, tp = confusion_matrix(grounds, predredictions).ravel().tolist()
        tpr = tp / (tp + fn) if (tp + fn) else 0
        fpr = fp / (fp + tn) if (fp + tn) else 0
        tnr = tn / (tn + fp) if (tn + fp) else 0
        fnr = fn / (fn + tp) if (fn + tp) else 0
        ppv = tp / (tp + fp) if (tp + fp) else 0
        npv = tn / (tn + fn) if (tn + fn) else 0
        acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0
        metric_df = pd.DataFrame({
            "cases": [c], "FPR": fpr, "TPR": tpr, "PPV": ppv, "NPV": npv,  "F1": f1,
            'adjusted ppv': tpr * 0.04 / (tpr * 0.04 + (fpr) * (1 - 0.04)) if (tpr * 0.04 + (fpr) * (1 - 0.04)) else 0,
            'adjusted npv': tnr * (1 - 0.04) / (tnr * (1 - 0.04) + fnr * 0.04)
        })
        if pw_results is None:
            pw_results = metric_df
        else:
            pw_results = pd.concat([pw_results, metric_df])
pw_results = pw_results.groupby("cases").mean()
pw_results.round(2)

0.65


,FPR,TPR,PPV,NPV,F1,adjusted ppv,adjusted npv
cases,,,,,,,
pec,0.15,0.5,0.46,0.88,0.47,0.14,0.98


In [44]:
cases = ["pec"]
pw_results = None
thresh = 0.89 #0.65, fmf 0.15, 
print(thresh)
for c in cases:
    for pw in range(1, 5):
        column = f"pw_{pw}"
        if "pec" == c:
            subset = pec_data[(~pec_data[column].isna())]
        else:
            subset = pec_data[(~pec_data[column].isna()) & ((pec_data[c] & pec_data["pec"]) | ~pec_data["pec"])]
        probabilities = np.array(list(subset[column])) + (subset["Past PEC"] * best_pp + best_fp * (subset["First Preg"] * (subset["Age at enrollment"] > best_age).astype(int)))
        predredictions = probabilities > thresh
        grounds = np.array(list(subset["label"]))

        f1 = f1_score(grounds, predredictions)
        tn, fp, fn, tp = confusion_matrix(grounds, predredictions).ravel().tolist()
        tpr = tp / (tp + fn) if (tp + fn) else 0
        fpr = fp / (fp + tn) if (fp + tn) else 0
        tnr = tn / (tn + fp) if (tn + fp) else 0
        fnr = fn / (fn + tp) if (fn + tp) else 0
        ppv = tp / (tp + fp) if (tp + fp) else 0
        npv = tn / (tn + fn) if (tn + fn) else 0
        acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0
        metric_df = pd.DataFrame({
            "cases": [c], "FPR": fpr, "TPR": tpr, "PPV": ppv, "NPV": npv,  "F1": f1,
            'adjusted ppv': tpr * 0.04 / (tpr * 0.04 + (fpr) * (1 - 0.04)) if (tpr * 0.04 + (fpr) * (1 - 0.04)) else 0,
            'adjusted npv': tnr * (1 - 0.04) / (tnr * (1 - 0.04) + fnr * 0.04)
        })
        if pw_results is None:
            pw_results = metric_df
        else:
            pw_results = pd.concat([pw_results, metric_df])
pw_results = pw_results.groupby("cases").mean()
pw_results.round(2)

0.89


,FPR,TPR,PPV,NPV,F1,adjusted ppv,adjusted npv
cases,,,,,,,
pec,0.05,0.26,0.57,0.84,0.36,0.21,0.97


### FMF metrics

In [45]:
cases = ["pec"]
pw_results = None
thresh = 0.15
print(thresh)
for c in cases:
    for pw in range(1, 5):
        column = f"pw_{pw}"
        if "pec" == c:
            subset = pec_data[(~pec_data[column].isna())]
        else:
            subset = pec_data[(~pec_data[column].isna()) & ((pec_data[c] & pec_data["pec"]) | ~pec_data["pec"])]
        probabilities = np.array(list(subset[column])) * 0 + (subset["fmf_prob"])
        predredictions = probabilities > thresh
        grounds = np.array(list(subset["label"]))

        f1 = f1_score(grounds, predredictions)
        tn, fp, fn, tp = confusion_matrix(grounds, predredictions).ravel().tolist()
        tpr = tp / (tp + fn) if (tp + fn) else 0
        fpr = fp / (fp + tn) if (fp + tn) else 0
        tnr = tn / (tn + fp) if (tn + fp) else 0
        fnr = fn / (fn + tp) if (fn + tp) else 0
        ppv = tp / (tp + fp) if (tp + fp) else 0
        npv = tn / (tn + fn) if (tn + fn) else 0
        acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0
        metric_df = pd.DataFrame({
            "cases": [c], "FPR": fpr, "TPR": tpr, "PPV": ppv, "NPV": npv,  "F1": f1,
            'adjusted ppv': tpr * 0.04 / (tpr * 0.04 + (fpr) * (1 - 0.04)) if (tpr * 0.04 + (fpr) * (1 - 0.04)) else 0,
            'adjusted npv': tnr * (1 - 0.04) / (tnr * (1 - 0.04) + fnr * 0.04)
        })
        if pw_results is None:
            pw_results = metric_df
        else:
            pw_results = pd.concat([pw_results, metric_df])
pw_results = pw_results.groupby("cases").mean()
pw_results.round(2)

0.15


,FPR,TPR,PPV,NPV,F1,adjusted ppv,adjusted npv
cases,,,,,,,
pec,0.04,0.11,0.4,0.82,0.17,0.11,0.96


In [46]:
cases = ["pec"]
pw_results = None
thresh = 0.08 
print(thresh)
for c in cases:
    for pw in range(1, 5):
        column = f"pw_{pw}"
        if "pec" == c:
            subset = pec_data[(~pec_data[column].isna())]
        else:
            subset = pec_data[(~pec_data[column].isna()) & ((pec_data[c] & pec_data["pec"]) | ~pec_data["pec"])]
        probabilities = np.array(list(subset[column])) * 0 + (subset["fmf_prob"])
        predredictions = probabilities > thresh
        grounds = np.array(list(subset["label"]))

        f1 = f1_score(grounds, predredictions)
        tn, fp, fn, tp = confusion_matrix(grounds, predredictions).ravel().tolist()
        tpr = tp / (tp + fn) if (tp + fn) else 0
        fpr = fp / (fp + tn) if (fp + tn) else 0
        tnr = tn / (tn + fp) if (tn + fp) else 0
        fnr = fn / (fn + tp) if (fn + tp) else 0
        ppv = tp / (tp + fp) if (tp + fp) else 0
        npv = tn / (tn + fn) if (tn + fn) else 0
        acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0
        metric_df = pd.DataFrame({
            "cases": [c], "FPR": fpr, "TPR": tpr, "PPV": ppv, "NPV": npv,  "F1": f1,
            'adjusted ppv': tpr * 0.04 / (tpr * 0.04 + (fpr) * (1 - 0.04)) if (tpr * 0.04 + (fpr) * (1 - 0.04)) else 0,
            'adjusted npv': tnr * (1 - 0.04) / (tnr * (1 - 0.04) + fnr * 0.04)
        })
        if pw_results is None:
            pw_results = metric_df
        else:
            pw_results = pd.concat([pw_results, metric_df])
pw_results = pw_results.groupby("cases").mean()
pw_results.round(2)

0.08


,FPR,TPR,PPV,NPV,F1,adjusted ppv,adjusted npv
cases,,,,,,,
pec,0.07,0.22,0.45,0.83,0.3,0.12,0.97


### Aspirin metrics

In [47]:
cases = ["pec"]
pw_results = None
thresh = 0.5
print(thresh)
for c in cases:
    for pw in range(1, 5):
        column = f"pw_{pw}"
        if "pec" == c:
            subset = pec_data[(~pec_data[column].isna())]
        else:
            subset = pec_data[(~pec_data[column].isna()) & ((pec_data[c] & pec_data["pec"]) | ~pec_data["pec"])]
        probabilities = np.array(list(subset[column])) * 0 + (subset["aspirin"])
        predredictions = probabilities > thresh
        grounds = np.array(list(subset["label"]))

        f1 = f1_score(grounds, predredictions)
        tn, fp, fn, tp = confusion_matrix(grounds, predredictions).ravel().tolist()
        tpr = tp / (tp + fn) if (tp + fn) else 0
        fpr = fp / (fp + tn) if (fp + tn) else 0
        tnr = tn / (tn + fp) if (tn + fp) else 0
        fnr = fn / (fn + tp) if (fn + tp) else 0
        ppv = tp / (tp + fp) if (tp + fp) else 0
        npv = tn / (tn + fn) if (tn + fn) else 0
        acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else 0
        metric_df = pd.DataFrame({
            "cases": [c], "FPR": fpr, "TPR": tpr, "PPV": ppv, "NPV": npv,  "F1": f1,
            'adjusted ppv': tpr * 0.04 / (tpr * 0.04 + (fpr) * (1 - 0.04)) if (tpr * 0.04 + (fpr) * (1 - 0.04)) else 0,
            'adjusted npv': tnr * (1 - 0.04) / (tnr * (1 - 0.04) + fnr * 0.04)
        })
        if pw_results is None:
            pw_results = metric_df
        else:
            pw_results = pd.concat([pw_results, metric_df])
pw_results = pw_results.groupby("cases").mean()
pw_results.round(2)

0.5


,FPR,TPR,PPV,NPV,F1,adjusted ppv,adjusted npv
cases,,,,,,,
pec,0.16,0.39,0.36,0.85,0.37,0.09,0.97
